# Individual homework project 2

You will apply statistical testing and correlation analysis to the ShopNow dataset.

For each question:

1. Run the appropriate test.
2. Report the test statistic and p-value (if applicable).
3. Interpret the result in plain English.
4. Explain what it means for the business.

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import scipy.stats as stats

In [21]:
customers = pd.read_csv('customers.csv')
transactions = pd.read_csv('transactions.csv')
web_sessions = pd.read_csv('web_sessions.csv')
campaign_touchpoints = pd.read_csv('campaign_touchpoints.csv')

### Specific questions

Q1. Do customers from the UK spend more per transaction than customers from Germany?

In [22]:
merged = transactions.merge(customers[['customer_id', 'country']], on='customer_id')
uk_spend = merged[merged['country'] == 'UK']['purchase_value']
de_spend = merged[merged['country'] == 'Germany']['purchase_value']

t_stat, p_val = stats.ttest_ind(uk_spend, de_spend, equal_var=False)
print(f"UK mean: {uk_spend.mean():.2f}, Germany mean: {de_spend.mean():.2f}")
print(f"t-stat: {t_stat:.4f}, p-val: {p_val:.4f}")

UK mean: 234.84, Germany mean: 239.03
t-stat: -0.8612, p-val: 0.3892


Result: t=−0.8612, p=0.3892 (UK mean: 234.84, Germany mean: 239.03).

Interpretation: The average transaction value was slightly higher in Germany, but the difference was not statistically significant (p>0.05). We do not have enough evidence to say that UK customers spend more per transaction.

Business Impact: The business should not change regional pricing based on this result alone. It can also compare sales volume, repeat purchases, and customer lifetime value in both countries.

Q2. Do discounted purchases differ in average value from non-discounted purchases?

In [23]:
disc_spend = transactions[transactions['discount_used'] == 1]['purchase_value']
non_disc_spend = transactions[transactions['discount_used'] == 0]['purchase_value']

t_stat, p_val = stats.ttest_ind(disc_spend, non_disc_spend, equal_var=False)
print(f"Disc mean: {disc_spend.mean():.2f}, Non-disc mean: {non_disc_spend.mean():.2f}")
print(f"t-stat: {t_stat:.4f}, p-val: {p_val:.4f}")

Disc mean: 235.61, Non-disc mean: 234.56
t-stat: 0.4013, p-val: 0.6882


Result: t=0.4013, p=0.6882 (Discounted mean: 235.61, non-discounted mean: 234.56).

Interpretation: The average values are close, and the difference is not statistically significant (p>0.05).

Business Impact: This test does not show that discounts increase or reduce the average transaction value. The business should also check sales volume, profit margins, and total revenue before deciding whether a discount campaign is successful.

Q3. Do male and female customers differ in average purchase value?

In [24]:
trans_gender = transactions.merge( customers[['customer_id', 'gender']],on='customer_id',)
male_spend = trans_gender.loc[trans_gender['gender'] == 'Male', 'purchase_value'].dropna()
female_spend = trans_gender.loc[trans_gender['gender'] == 'Female', 'purchase_value'].dropna()
t_stat, p_val = stats.ttest_ind(male_spend, female_spend, equal_var=False)
print(f"Male mean: {male_spend.mean():.2f}, Female mean: {female_spend.mean():.2f}")
print(f"t-stat: {t_stat:.4f}, p-val: {p_val:.4f}")

Male mean: 234.81, Female mean: 235.94
t-stat: -0.3875, p-val: 0.6984


Result: t=−0.3875, p=0.6984 (Male mean: 234.81, female mean: 235.94).

Interpretation: The difference in average purchase value was not statistically significant (p>0.05).

Business Impact: The business should avoid assuming that one gender spends more based on these results. Customer behaviour, purchase history, and product interests may be more useful when planning marketing.

Q4. Do customers acquired via email spend more than those from social channels?

In [25]:
merged = transactions.merge(customers[['customer_id', 'acquisition_channel']], on='customer_id')
email_spend = merged[merged['acquisition_channel'] == 'email']['purchase_value']
social_spend = merged[merged['acquisition_channel'] == 'social']['purchase_value']

t_stat, p_val = stats.ttest_ind(email_spend, social_spend, equal_var=False)
print(f"Email mean: {email_spend.mean():.2f}, Social mean: {social_spend.mean():.2f}")
print(f"t-stat: {t_stat:.4f}, p-val: {p_val:.4f}")

Email mean: 235.99, Social mean: 232.19
t-stat: 0.9873, p-val: 0.3235


Result: t=0.9873, p=0.3235 (Email mean: 235.99, social mean: 232.19).

Interpretation: Customers acquired through email had a slightly higher average transaction value, but the difference was not statistically significant (p>0.05).

Business Impact: The business should compare channel cost, number of customers acquired, conversion rate, and repeat purchases before deciding how to split its marketing budget.

Q5. Is device type associated with conversion?

In [26]:
contingency_device = pd.crosstab(web_sessions['device_type'], web_sessions['converted'])
chi2, p_val, dof, ex = stats.chi2_contingency(contingency_device)
conversion_rates = (web_sessions.groupby('device_type')['converted'].mean().mul(100).round(2))
print(contingency_device)
print("\nConversion rate by device (%):")
print(conversion_rates)
print(f"\nChi2: {chi2:.4f}, p-val: {p_val:.4f}")

converted        0     1
device_type             
desktop      20020  3068
mobile       20213  3042
tablet       20244  3013

Conversion rate by device (%):
device_type
desktop    13.29
mobile     13.08
tablet     12.96
Name: converted, dtype: float64

Chi2: 1.1502, p-val: 0.5626


Result: Chi2 = 1.1502, p-value = 0.5626. Desktop conversion was 13.29%, mobile was 13.08%, and tablet was 12.96%.

Interpretation: There was no statistically significant association between device type and conversion because the p-value was greater than 0.05.

Business Impact: Conversion rates were very similar across the three devices. The business can still check user feedback and checkout problems to identify possible improvements.

Q6. Is acquisition channel associated with conversion?

In [27]:
customers_clean = customers.copy()
customers_clean['acquisition_channel'] = (customers_clean['acquisition_channel'].astype('string').str.strip().str.lower())
merged = web_sessions.merge(customers_clean[['customer_id', 'acquisition_channel']],on='customer_id',)  
contingency_source = pd.crosstab(merged['acquisition_channel'], merged['converted'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_source)

print(contingency_source)
print(f"Chi2: {chi2:.4f}, p-val: {p_val:.4f}")

converted                0     1
acquisition_channel             
email                13525  2048
organic              11079  1673
paid_search          13209  1914
referral             11563  1746
social               11545  1798
Chi2: 4.3101, p-val: 0.3657


Result: Chi-square = 4.3101, p-value = 0.3657.

Interpretation: The p-value is greater than 0.05, so there is no statistically significant association between acquisition channel and conversion. Although the conversion counts differ across channels, the test does not provide enough evidence that conversion is associated with the acquisition channel.

Business Impact: The business should not choose a marketing channel based on these results alone. It should also compare conversion rates, advertising costs, and customer value to understand which channels perform better.

Q7. Is product category associated with discount usage?

In [28]:
contingency_product_category = pd.crosstab(transactions['product_category'], transactions['discount_used'])
chi2, p_val, dof, ex = stats.chi2_contingency(contingency_product_category)
print(contingency_product_category)
print(f"Chi2: {chi2:.4f}, p-val: {p_val:.4f}")

discount_used        0     1
product_category            
beauty            5305  2281
electronics       5212  2212
fashion           5326  2255
home              5210  2278
sports            5225  2279
toys              5196  2221
Chi2: 1.4622, p-val: 0.9174


Result: chi2=1.4622, p=0.9174.

Interpretation: There is no statistically significant association between product category and discount usage (p>0.05).

Business Impact: This dataset does not show a clear difference in discount usage between categories. The business can continue to monitor discount rates and profit margins by category to make sure promotions are useful.

Q8. Is time on site correlated with purchase value?

In [29]:
cust_web = web_sessions.groupby('customer_id')['time_on_site'].mean().reset_index()
cust_trans = transactions.groupby('customer_id')['purchase_value'].mean().reset_index()
merged_cust = cust_web.merge(cust_trans, on='customer_id').dropna()

corr_time_on_site, p_time_on_site = stats.pearsonr(merged_cust['time_on_site'], merged_cust['purchase_value'])
print(f"r: {corr_time_on_site:.4f}, p-val: {p_time_on_site:.4f}")

r: 0.0094, p-val: 0.3501


Result: r=0.0094, p=0.3501.

Interpretation: There is no statistically significant linear correlation between a customer's average time on site and average purchase value (p>0.05). The correlation is also very close to zero.

Business Impact: Spending more time on the site does not necessarily mean a customer will make a higher-value purchase. The business should focus on helping customers find products and complete purchases easily.

Q9. Is number of pages viewed correlated with purchase value?

In [30]:
cust_web_pages = web_sessions.groupby('customer_id')['pages_viewed'].mean().reset_index()
merged_cust_pages = cust_web_pages.merge(cust_trans, on='customer_id').dropna()

corr_pages_viewed, p_pages_viewed = stats.pearsonr(merged_cust_pages['pages_viewed'], merged_cust_pages['purchase_value'])
print(f"r: {corr_pages_viewed:.4f}, p-val: {p_pages_viewed:.4f}")

r: -0.0183, p-val: 0.0689


Result: r=−0.0183, p=0.0689.

Interpretation: There is no statistically significant linear correlation between average pages viewed and average purchase value at the 5% level (p>0.05).

Business Impact: More page views do not necessarily mean a higher-value purchase.

Q10. Is time on site correlated with pages viewed?

In [31]:
session_metrics = web_sessions[['time_on_site', 'pages_viewed']].dropna()
corr, p = stats.pearsonr(session_metrics['time_on_site'],session_metrics['pages_viewed'])
print(f"r: {corr:.4f}, p-val: {p:.4f}")

r: 0.0045, p-val: 0.2293


Result: r=0.0045, p=0.2293.

Interpretation: There is no statistically significant linear correlation between time on site and pages viewed (p>0.05). The correlation is very close to zero.

Business Impact: Longer sessions do not necessarily involve more pages. The business should look at other measures, such as completed checkouts and customer feedback, to understand how people use the website.

### Challenges

C1. Explore whether high-spending customers behave differently in terms of session time.

Define “high spending” yourself.

In [32]:
cust_total_spend = transactions.groupby('customer_id')['purchase_value'].sum().reset_index()
threshold = cust_total_spend['purchase_value'].quantile(0.80)
cust_total_spend['is_high_spender'] = cust_total_spend['purchase_value'] >= threshold
cust_session_metrics = web_sessions.groupby('customer_id')['time_on_site'].mean().reset_index()
cust_behavior = pd.merge(cust_session_metrics, cust_total_spend, on='customer_id')

high_spenders_time = cust_behavior[cust_behavior['is_high_spender'] == True]['time_on_site']
low_spenders_time = cust_behavior[cust_behavior['is_high_spender'] == False]['time_on_site']

t_stat, p_val = stats.ttest_ind(high_spenders_time, low_spenders_time, equal_var=False)
print(f"High Spenders Mean Session Time: {high_spenders_time.mean():.2f} seconds")
print(f"Normal Spenders Mean Session Time: {low_spenders_time.mean():.2f} seconds")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")

High Spenders Mean Session Time: 302.15 seconds
Normal Spenders Mean Session Time: 300.48 seconds
t-statistic: 1.2875, p-value: 0.1980


Result: t=1.2875, p=0.1980 (High-spender mean session time: 302.15 seconds; other customers: 300.48 seconds).

Interpretation: The difference in average session time was not statistically significant (p>0.05). High spenders were defined as customers at or above the 80th percentile of total spending.

Business Impact: High spenders do not clearly spend more time on the site. The business could also study purchase frequency and repeat visits to understand valuable customers.

C2. Find a relationship (difference or correlation) that you believe is meaningful for business strategy.

Explain why.

In [33]:
refund_table = pd.crosstab(transactions['product_category'],transactions['is_refunded'].fillna(0))
chi2, p_val, dof, expected = stats.chi2_contingency(refund_table)
refund_rates = (transactions.assign(refunded=transactions['is_refunded'].fillna(0)).groupby('product_category')['refunded'].mean().mul(100).round(2).rename('refund_rate_percent'))
print("Refund counts by product category:")
print(refund_table)
print("\nRefund rate by product category (%):")
print(refund_rates)
print(f"\nChi2-statistic: {chi2:.4f}, p-value: {p_val:.4f}")

Refund counts by product category:
is_refunded          0    1
product_category           
beauty            6986  600
electronics       6812  612
fashion           6957  624
home              6848  640
sports            6919  585
toys              6846  571

Refund rate by product category (%):
product_category
beauty         7.91
electronics    8.24
fashion        8.23
home           8.55
sports         7.80
toys           7.70
Name: refund_rate_percent, dtype: float64

Chi2-statistic: 5.2663, p-value: 0.3842


Result: The chi2= 5.2663, p= 0.3842.

Interpretation: The p-value was greater than 0.05, so there was no statistically significant association between product category and refund status. This means the analysis did not find enough evidence that refund status differs by product category.

Business Impact: The business should continue monitoring refund rates across product categories. If refund rates increase in a particular category, it can investigate possible reasons such as product quality, delivery problems, or customer expectations.

c3. Identify a statistically significant result that may not be practically meaningful.

Explain why statistical significance ≠ business importance.

In [34]:
bounce_conversion = pd.crosstab(web_sessions['bounce'], web_sessions['converted'])
chi2, p_val, dof, expected = stats.chi2_contingency(bounce_conversion)
n = bounce_conversion.to_numpy().sum()
cramers_v = np.sqrt(chi2 / n)
conversion_rates = (web_sessions.groupby('bounce')['converted'].mean().mul(100).round(2))
print("Bounce vs conversion table:")
print(bounce_conversion)
print(f"\nChi2-statistic: {chi2:.4f}")
print(f"P-value: {p_val:.10f}")
print(f"Cramer's V: {cramers_v:.4f}")
print("Conversion rate by bounce status (%):")
print(conversion_rates)

Bounce vs conversion table:
converted      0     1
bounce                
0          49374  7680
1          11547  1499

Chi2-statistic: 36.0691
P-value: 0.0000000019
Cramer's V: 0.0227
Conversion rate by bounce status (%):
bounce
0    13.46
1    11.49
Name: converted, dtype: float64


Result: The relationship between bounce status and conversion was statistically significant (p-value < 0.05), but Cramér's V was very small (about 0.023).

Interpretation: Sessions with a bounce had a lower conversion rate in this dataset. However, the relationship was very weak. The large number of sessions may help explain why the result was statistically significant even though the difference was small.

Business Impact: This shows why statistical significance does not always mean business importance. A result can be statistically significant but have little practical effect on conversions or revenue. The business should consider the actual size of the difference, the potential cost and benefit of improvements, and whether testing confirms that a change improves performance. The observed association does not prove that bouncing causes lower conversion.